## Scratchpad

In [1]:
from lcn.core.model import LCN

# Load an example LCN
filename = "/Users/radu/git/IBM/LCN/examples/d4_biting.lcn"

lcn = LCN()
lcn.from_lcn(file_name=filename)

G1 = lcn.build_primal_graph(formula_labels=True)
G2 = lcn.build_structure_graph()

Parsed LCN format with 4 sentences.
Build the LCN's primal graph.
Build the LCN's structure graph.
Processing sentence: A1: 0.4 <= P(A) <= 0.6
Processing sentence: B1: 0.1 <= P(B | A) <= 0.9
Processing sentence: C1: 0.1 <= P(C | A) <= 0.9
Processing sentence: BC: 0.0 <= P(B and C) <= 0.05
Replacing the following bidirected edges with undirected edges: []
Build the LCN's independence assumptions (LMC).
Processing sentence: A1: 0.4 <= P(A) <= 0.6
Processing sentence: B1: 0.1 <= P(B | A) <= 0.9
Processing sentence: C1: 0.1 <= P(C | A) <= 0.9
Processing sentence: BC: 0.0 <= P(B and C) <= 0.05
Replacing the following bidirected edges with undirected edges: []


In [2]:
import gravis as gv

fig1 = gv.d3(G1, node_label_data_source='label', show_node_label_border=True, edge_curvature=0.0)
fig1

ModuleNotFoundError: No module named 'pkg_resources'

In [3]:
G = G2.to_networkx_digraph()
fig = gv.d3(G, node_label_data_source='id', show_node_label_border=True, edge_curvature=0.0)
fig

NameError: name 'gv' is not defined

In [4]:
# check if the LCN is a chain graph 
ok = lcn.is_chain_graph()
print(f"Is the LCN a chain graph? {ok}")

Simplifying the structure by replacing the undirected cliques
Is the LCN a chain graph? True


In [5]:
# get the families of each node in the chain graph
families = lcn.process_chain_graph()
print("Families of each node:")
for family in families:
    node = family["child"]
    print(f"{node}: {family}")

Child: A, Parents: [], Scope: ['A']
Adding sentence A1 with scope {'A'}.
Child: B, Parents: ['A'], Scope: ['B', 'A']
Adding sentence A1 with scope {'A'}.
Adding sentence B1 with scope {'B', 'A'}.
Child: C, Parents: ['A'], Scope: ['C', 'A']
Adding sentence A1 with scope {'A'}.
Adding sentence C1 with scope {'C', 'A'}.
Families of each node:
A: {'child': 'A', 'parents': [], 'sentences': ['A1']}
B: {'child': 'B', 'parents': ['A'], 'sentences': ['A1', 'B1']}
C: {'child': 'C', 'parents': ['A'], 'sentences': ['A1', 'C1']}


In [6]:
# Build the symbolic chain-graph factorization (one factor P(child | parents)
# per family). The class moved to lcn.inference.marginal.cn and was renamed.
from lcn.inference.marginal.cn.factorization import ChainGraphFactorization

fact = ChainGraphFactorization(lcn)
factors = fact.build()

print("Symbolic chain-graph factors P(child | parents):")
for f in factors:
    print(f"  P({f['child']} | {f['parents']})  scope={f['scope']}  "
          f"sentences={f['sentences']}")


Symbolic chain-graph factors P(child | parents):
  P(A | [])  scope=['A']  sentences=['A1']
  P(B | ['A'])  scope=['B', 'A']  sentences=['A1', 'B1']
  P(C | ['A'])  scope=['C', 'A']  sentences=['A1', 'C1']


In [7]:
# Compile the LCN to a credal network and enumerate the extreme points of each
# local credal set. `method`: "linear" (in-scope-sentence LP, default) or
# "linear-tight" (adds the scope-restricted LMC equalities -- scheme D1).
# `solver`: "ipopt" (local, default) or "scip" (certified global -- scheme D3).
# Set verbosity=2 to stream the ipopt/scip solver progress.
from lcn.inference.marginal.cn.vertices import CredalNetworkVertices

cnv = CredalNetworkVertices.from_lcn(lcn, method="linear", verbosity=1)


Symbolic factor: A <-- []
  parents_lst: []
  scope: ['A']
Symbolic factor: B <-- ['A']
  parents_lst: ['A']
  scope: ['B', 'A']
Symbolic factor: C <-- ['A']
  parents_lst: ['A']
  scope: ['C', 'A']
[CredalNetwork] Chain-graph factorization produced 3 symbolic factors.
[CredalNetworkVertices] Lower BN CPTs:
  A: 
  A                |
0        |1        |
---------|---------|
 0.4000  | 0.4000  |

  B: 
      ||  B                |
A     ||0        |1        |
------||---------|---------|
0     || 0.0000  | 0.0000  |
1     || 0.1000  | 0.1000  |

  C: 
      ||  C                |
A     ||0        |1        |
------||---------|---------|
0     || 0.0000  | 0.0000  |
1     || 0.1000  | 0.1000  |

[CredalNetworkVertices] Upper BN CPTs:
  A: 
  A                |
0        |1        |
---------|---------|
 0.6000  | 0.6000  |

  B: 
      ||  B                |
A     ||0        |1        |
------||---------|---------|
0     || 1.0000  | 1.0000  |
1     || 0.9000  | 0.9000  |

  C: 
      ||

In [8]:
# Credal Variable Elimination now computes ALL posterior marginals in one run():
# it loops the per-target bucket elimination over the network nodes and returns
# {name -> (lower, upper)}. It also records the running times and flags a
# DEGENERATE (all-vacuous [0,1]) solution.
from lcn.inference.marginal.cn.cve import CredalVE

cve = CredalVE(cnv=cnv)
results = cve.run(evidence={}, verbosity=1)

print("\nSingleton-atom marginals P(atom=1):")
for atom in sorted(cve.singleton_marginals):
    lo, hi = cve.singleton_marginals[atom]
    print(f"  P({atom}=1) in [{lo:.6f}, {hi:.6f}]")

print(f"\nbuild time={cve.build_time:.4f}s, "
      f"elimination time={cve.elimination_time:.4f}s, "
      f"total time={cve.total_time:.4f}s")
print(f"degenerate (all marginals vacuous [0,1])? {cve.degenerate}")


[CredalVE] Computing all marginals (coupling=off, evidence={})
[CredalVE] Query: A
[CredalVE] Evidence: {}
[CredalVE] Elimination order (topological): ['C', 'B']
[CredalVE] Initial potentials: 3, total functions: 10
[CredalVE] Query: B
[CredalVE] Evidence: {}
[CredalVE] Elimination order (topological): ['A', 'C']
[CredalVE] Initial potentials: 3, total functions: 10
[CredalVE] Query: C
[CredalVE] Evidence: {}
[CredalVE] Elimination order (topological): ['A', 'B']
[CredalVE] Initial potentials: 3, total functions: 10
[CredalVE] Singleton marginals P(atom=1):
  P(A=1): [0.400000, 0.600000]
  P(B=1): [0.040000, 0.960000]
  P(C=1): [0.040000, 0.960000]
[CredalVE] Running times (seconds):
  build time:       4.0634
  running time:     0.0070
  total time:       4.0704

Singleton-atom marginals P(atom=1):
  P(A=1) in [0.400000, 0.600000]
  P(B=1) in [0.040000, 0.960000]
  P(C=1) in [0.040000, 0.960000]

build time=4.0634s, elimination time=0.0070s, total time=4.0704s
degenerate (all marginal

In [9]:
# Tighter / exact options (schemes D4 and D5), opt-in via `coupling`:
#   "off"          -- bounds the strong extension (default, above);
#   "cross-family" -- D4, forbids vertex combinations that violate a
#                     cross-family LCN sentence/LMC assertion;
#   "d5"           -- D5, the junction-tree EXACT NLP (defaults to the certified
#                     global SCIP backend). Exact for both marginal and
#                     conditional queries.
cve_d5 = CredalVE(cnv=cnv)
cve_d5.run(evidence={}, coupling="d5", verbosity=1)

print("\nExact marginals via D5:")
for atom in sorted(cve_d5.singleton_marginals):
    lo, hi = cve_d5.singleton_marginals[atom]
    print(f"  P({atom}=1) in [{lo:.6f}, {hi:.6f}]")


[CredalVE] Computing all marginals (coupling=d5, solver=scip, evidence={})
[D5] junction tree: 2 clusters, max cluster 3 atoms (n=3 atoms total)
[D5] junction tree: 2 clusters, max cluster 3 atoms (n=3 atoms total)
[D5] junction tree: 2 clusters, max cluster 3 atoms (n=3 atoms total)
[CredalVE] Singleton marginals P(atom=1):
  P(A=1): [0.400000, 0.600000]
  P(B=1): [0.040000, 0.960000]
  P(C=1): [0.040000, 0.960000]
[CredalVE] D5 exact=True, max cluster=3 atoms
[CredalVE] Running times (seconds):
  build time:       4.0634
  running time:     0.4589
  total time:       4.5223

Exact marginals via D5:
  P(A=1) in [0.400000, 0.600000]
  P(B=1) in [0.040000, 0.960000]
  P(C=1) in [0.040000, 0.960000]
